# 5-HT2A Docking + MD (Notebook 2 of 2)

Dock → parameterise → merge → MD for the three racetams at the **5-HT2A receptor**, reusing the equilibrated membrane system built in Notebook 1. This is the same validated pipeline used for the D2 racetam runs (automatic heavy-atom mapping, NAGL charges, ParmEd-level merge, membrane barostat), retargeted to 5-HT2A.

**Verified inputs from Notebook 1** (in `/workspace/5ht2a_system/`):
| File | Role |
|---|---|
| `ht2a_gmx_top.pkl` | 5-HT2A + DPPC ParmEd structure (CHARMM36, from MemProtMD) |
| `ht2a_final_equilibrated.xml` | equilibrated state |
| `ht2a_equilibrated_protein.pdb` | receptor for docking |

**Verified pocket (chain A, from the risperidone pose in 6A93):**
- Orthosteric centre **(13.75, −0.74, 60.46) Å**, docking box **22 Å** (slightly larger than D2's 20 Å to also sample the 5-HT2A side-extended cavity)
- Aminergic anchor **ASP155 (D3.32)** — the 5-HT2A equivalent of D2's ASP114; watch whether the racetams (which lack a basic amine) engage it
- Toggle switch **TRP336 (W6.48)** — run the same rotamer analysis as D2
- Aromatic cage: TRP151, PHE243, PHE332, PHE339, PHE340, TYR370; serines SER159/SER242

**Run one ligand at a time** by setting the three variables in Cell 2. Start with levetiracetam.

## ⚠ Residue renumbering (Amber tleap)

The Amber-parameterised 5-HT2A system renumbered residues sequentially (0–283). Verified mapping of key residues (original → new):

| Original | New | Residue |
|---|---|---|
| ASP155 (D3.32 anchor) | **86** | ASP |
| TRP336 (W6.48 toggle) | **220** | TRP |
| PHE332 (F6.44) | **216** | PHE |
| SER242 (S5.46) | **173** | SER |
| SER159 | **90** | SER |

The analysis cell tracks the renumbered **residue 86** for the ASP155 aminergic-anchor contact. Report results using the original 5-HT2A numbers (ASP155, W336) in the manuscript; the new numbers are an internal Amber artifact.

**Force field note:** this 5-HT2A system uses **Amber ff19SB / Lipid21 / TIP3P** (D2 used CHARMM36). State as a builder-driven choice in Methods; both validated, each system internally consistent.

---
## Part 1 — Environment and inputs

In [ ]:
# Cell 1: Imports
import sys, os, time, pickle
from pathlib import Path
import numpy as np
import openmm
from openmm import (XmlSerializer, LangevinMiddleIntegrator, MonteCarloMembraneBarostat,
                    Platform, CustomExternalForce, Vec3)
from openmm import unit as u
from openmm import app
import parmed as pmd
from openff.toolkit import Molecule, ForceField as OFFForceField
from rdkit import Chem
from rdkit.Chem import AllChem, rdDetermineBonds
print("OpenMM", openmm.__version__, "| ParmEd", pmd.__version__)
plats=[Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
assert "CUDA" in plats, plats
print("✓ CUDA available")

In [ ]:
# Cell 2: SELECT LIGAND (edit these three lines per run)
LIGANDS = {
  "levetiracetam": dict(smiles="CC[C@@H](N1CCCC1=O)C(N)=O",        n_heavy=12, lig="LEV"),
  "brivaracetam":  dict(smiles="CCC[C@@H]1CC(=O)N(C1)[C@@H](CC)C(=O)N", n_heavy=15, lig="BRV"),
  "piracetam":     dict(smiles="NC(=O)CN1CCCC1=O",                 n_heavy=10, lig="PIR"),
}
DRUG = "levetiracetam"          # <-- change to brivaracetam / piracetam for later runs
cfg = LIGANDS[DRUG]
SMILES, N_HEAVY, LIGNAME = cfg["smiles"], cfg["n_heavy"], cfg["lig"]
PREFIX = f"ht2a_{LIGNAME.lower()}"
print(f"Ligand: {DRUG}  SMILES {SMILES}  heavy {N_HEAVY}  resname {LIGNAME}  prefix {PREFIX}")

WORKDIR = Path("/workspace/5ht2a_system"); os.chdir(WORKDIR)
for f in ["ht2a_gmx_top.pkl","ht2a_final_equilibrated.xml","ht2a_equilibrated_protein.pdb"]:
    assert (WORKDIR/f).exists(), f"Missing {f} — run Notebook 1 first"
print("✓ Inputs present")

POCKET_CENTER = [13.75, -0.74, 60.46]   # verified orthosteric centre, chain A
BOX_SIZE = [22.0, 22.0, 22.0]           # covers orthosteric + side-extended cavity

---
## Part 2 — Build ligand and dock (orthosteric + side cavity in one box)

In [ ]:
# Cell 3: Build ligand from SMILES -> SDF -> PDBQT (Meeko)
mol = Chem.MolFromSmiles(SMILES); assert mol and mol.GetNumHeavyAtoms()==N_HEAVY
mol_h = Chem.AddHs(mol); AllChem.EmbedMolecule(mol_h, randomSeed=42); AllChem.MMFFOptimizeMolecule(mol_h)
LIG_SDF = WORKDIR/f"{PREFIX}_ligand.sdf"; Chem.MolToMolFile(mol_h, str(LIG_SDF))
from meeko import MoleculePreparation
try:
    from meeko import PDBQTWriterLegacy; _W=True
except Exception: _W=False
LIG_PDBQT = WORKDIR/f"{PREFIX}_ligand.pdbqt"
prep = MoleculePreparation(); setups = prep.prepare(mol_h); setup = setups[0] if isinstance(setups,list) else setups
if _W:
    s,ok,err = PDBQTWriterLegacy.write_string(setup); assert ok, err; LIG_PDBQT.write_text(s)
else:
    LIG_PDBQT.write_text(setup.write_pdbqt_string())
print(f"✓ {LIG_PDBQT.name}")

In [ ]:
import os, shutil, time
os.environ["PATH"] = "/workspace/miniforge3/envs/mddock/bin:" + os.environ["PATH"]
print(shutil.which("obabel"))

In [ ]:
# Find the actual orthosteric pocket center from key residues in the equilibrated structure
# ASP155 (renumbered to res 86) is the anchor residue
from openbabel import openbabel as ob

# Parse receptor to find pocket residues
rec_coords = []
with open("ht2a_equilibrated_protein.pdb") as f:
    for line in f:
        if line.startswith("ATOM"):
            resSeq = int(line[22:26].strip())
            atom_name = line[12:16].strip()
            # Key pocket residues (renumbered): ASP86, TRP220, PHE216, SER173, SER90
            if resSeq in [86, 90, 173, 216, 220]:
                x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
                rec_coords.append([x, y, z])

import numpy as np
pocket_center = np.mean(rec_coords, axis=0).tolist()
print(f"Recalculated pocket center: {pocket_center}")
print(f"Original (wrong) center:    {POCKET_CENTER}")

# Update the pocket center
POCKET_CENTER = [round(c, 2) for c in pocket_center]
print(f"Using new center: {POCKET_CENTER}")


In [ ]:
from pathlib import Path
REC_PDBQT = Path("/workspace/5ht2a_system/ht2a_equilibrated_protein.pdbqt")
print(REC_PDBQT.exists(), REC_PDBQT)

In [ ]:
# Delete old docked file
DOCKED = WORKDIR / f"{PREFIX}_docked.pdbqt"
DOCKED.unlink(missing_ok=True)

from vina import Vina
v = Vina(sf_name="vina", cpu=0, verbosity=1)
v.set_receptor(str(REC_PDBQT))
v.set_ligand_from_file(str(LIG_PDBQT))
v.compute_vina_maps(center=POCKET_CENTER, box_size=BOX_SIZE)
print(f"Docking {DRUG} at center {POCKET_CENTER} (exhaustiveness=32) ...")
v.dock(exhaustiveness=32, n_poses=9)
energies = v.energies(n_poses=9)
best_score = energies[0][0]
for i, e in enumerate(energies):
    print(f"  Pose {i+1}: {e[0]:.3f}")
print(f"🎯 Best {DRUG} pose: {best_score:.3f} kcal/mol")
v.write_poses(str(DOCKED), n_poses=9, overwrite=True)
print(f"✓ {DOCKED.name}")


---
## Part 3 — Parameterise at docked pose (automatic mapping + NAGL)

In [ ]:
# Cell 5: Parse best pose heavy atoms
def parse_pose(p):
    coords,elems=[],[]
    for line in open(p):
        if line.startswith("ENDMDL"): break
        if line.startswith(("ATOM","HETATM")):
            nm=line[12:16].strip()
            if nm.startswith("H"): continue
            el=line[76:78].strip()
            el={"OA":"O","NA":"N","HD":"H","SA":"S","A":"C"}.get(el, el[:1] if el else "C")
            if el=="H": continue
            coords.append([float(line[30:38]),float(line[38:46]),float(line[46:54])]); elems.append(el)
    return np.array(coords), elems
dcoords, delems = parse_pose(DOCKED)
print(f"Docked heavy atoms: {len(dcoords)} {delems}")
assert len(dcoords)==N_HEAVY

In [ ]:
# Cell 6: Automatic connectivity mapping (no manual table)
def conn_only(m):
    em=Chem.RWMol(m)
    for b in em.GetBonds(): b.SetBondType(Chem.BondType.SINGLE); b.SetIsAromatic(False)
    for a in em.GetAtoms(): a.SetIsAromatic(False); a.SetFormalCharge(0); a.SetNoImplicit(True)
    return em.GetMol()
xyz=f"{len(dcoords)}\n\n"+"".join(f"{e} {x:.4f} {y:.4f} {z:.4f}\n" for e,(x,y,z) in zip(delems,dcoords))
dmol=Chem.MolFromXYZBlock(xyz); rdDetermineBonds.DetermineConnectivity(dmol)
templ=Chem.MolFromSmiles(SMILES)
match=conn_only(dmol).GetSubstructMatch(conn_only(templ))
assert len(match)==N_HEAVY, f"mapping failed {len(match)}/{N_HEAVY}"
for ti in range(len(match)): assert templ.GetAtomWithIdx(ti).GetSymbol()==delems[match[ti]]
print(f"✓ Automatic mapping {len(match)}/{N_HEAVY}, element-consistent")

In [ ]:
# Cell 7: Transplant coords, relax H, NAGL charges, ParmEd
th=Chem.AddHs(templ); AllChem.EmbedMolecule(th,randomSeed=42); AllChem.MMFFOptimizeMolecule(th)
conf=th.GetConformer(); heavy=[a.GetIdx() for a in th.GetAtoms() if a.GetAtomicNum()>1]
for ti,mi in enumerate(heavy):
    x,y,z=dcoords[match[ti]]; conf.SetAtomPosition(mi,(float(x),float(y),float(z)))
mp=AllChem.MMFFGetMoleculeProperties(th); ff=AllChem.MMFFGetMoleculeForceField(th,mp)
for i in heavy: ff.AddFixedPoint(i)
ff.Minimize(maxIts=500)
ligand_off=Molecule.from_rdkit(th)
from openff.toolkit.utils.nagl_wrapper import NAGLToolkitWrapper
NAGLToolkitWrapper().assign_partial_charges(molecule=ligand_off,
    partial_charge_method="openff-gnn-am1bcc-0.1.0-rc.3.pt")
print(f"✓ NAGL charges (sum {sum(c.m for c in ligand_off.partial_charges):.3f} e)")
off_ff=OFFForceField("openff-2.1.0.offxml")
inter=off_ff.create_interchange(topology=ligand_off.to_topology(), charge_from_molecules=[ligand_off])
from openff.units import unit as ou
inter.box=np.eye(3)*5.0*ou.nanometer
inter.to_top(f"_t_{PREFIX}.top"); inter.to_gro(f"_t_{PREFIX}.gro")
ligand_struct=pmd.load_file(f"_t_{PREFIX}.top", xyz=f"_t_{PREFIX}.gro")
ligand_struct.coordinates=np.array([list(conf.GetAtomPosition(i)) for i in range(th.GetNumAtoms())])
for r in ligand_struct.residues: r.name=LIGNAME
print(f"✓ Ligand ParmEd: {len(ligand_struct.atoms)} atoms, centroid {ligand_struct.coordinates.mean(0).round(2)}")

---
## Part 4 — Merge, build, equilibrate, produce

In [ ]:
print('ligand_struct' in dir(), 'ht2a' in dir(), 'combined' in dir())

In [ ]:
print('ligand_struct' in dir())

In [ ]:
# Cell 8: Load equilibrated 5-HT2A and merge (workaround for ParmEd copy bug)
import copy
with open("ht2a_gmx_top.pkl", "rb") as f:
    ht2a = pickle.load(f)

with open("ht2a_final_equilibrated.xml") as f:
    st = XmlSerializer.deserialize(f.read())

pos = st.getPositions(asNumpy=True)
box = st.getPeriodicBoxVectors(asNumpy=True)
assert len(pos) == len(ht2a.atoms)
ht2a.coordinates = pos.value_in_unit(u.angstrom)
ht2a.box = [box[0][0].value_in_unit(u.angstrom), box[1][1].value_in_unit(u.angstrom),
            box[2][2].value_in_unit(u.angstrom), 90, 90, 90]

# Fix: Initialize _ncopies if missing (ParmEd bug workaround)
if not hasattr(ht2a, '_ncopies'):
    ht2a._ncopies = 0

combined = ht2a + ligand_struct
combined.box = ht2a.box

from collections import Counter
assert Counter(r.name for r in combined.residues).get(LIGNAME, 0) == 1
print(f"✓ Merged: {len(combined.atoms):,} atoms, {LIGNAME} present")


In [ ]:
# Cell 9: OpenMM system + barostat + simulation
system=combined.createSystem(nonbondedMethod=app.PME, nonbondedCutoff=1.0*u.nanometer,
    constraints=app.HBonds, rigidWater=True, removeCMMotion=True, ewaldErrorTolerance=5e-4)
system.addForce(MonteCarloMembraneBarostat(1.0*u.bar,0.0*u.bar*u.nanometer,310*u.kelvin,
    MonteCarloMembraneBarostat.XYIsotropic, MonteCarloMembraneBarostat.ZFree, 25))
integ=LangevinMiddleIntegrator(310*u.kelvin,1.0/u.picosecond,0.002*u.picosecond)
sim=app.Simulation(combined.topology, system, integ, Platform.getPlatformByName("CUDA"),
    {"Precision":"mixed","DeviceIndex":"0"})
sim.context.setPositions(combined.positions)
bn=np.array(combined.box[:3])/10.0
sim.context.setPeriodicBoxVectors(Vec3(bn[0],0,0)*u.nanometer,Vec3(0,bn[1],0)*u.nanometer,Vec3(0,0,bn[2])*u.nanometer)
print(f"✓ Simulation ready, {system.getNumParticles():,} atoms")

---
## Part 5 — Quick analysis (ASP155 + W336, the 5-HT2A pocket)

In [ ]:
import mdtraj as mdt, numpy as np
traj = mdt.load("ht2a_lev_production.dcd", top="ht2a_lev_minimized.pdb")

# Diagnose: how does MDTraj see the aspartates?
asps = [r for r in traj.topology.residues if r.name in ("ASP","ASH")]
print("ASP residues MDTraj sees (resSeq, index):")
for r in asps:
    print(f"  resSeq={r.resSeq}, index={r.index}, name={r.name}")

In [ ]:
import mdtraj as mdt, numpy as np
traj = mdt.load("ht2a_lev_production.dcd", top="ht2a_lev_minimized.pdb")

# The aminergic anchor: original ASP155. Find ALL aspartate carboxyl oxygens,
# then identify which ASP is closest to the ligand on average (that's the binding one).
lig = traj.topology.select("resname LEV and (type O or type N)")
if len(lig)==0:  # ligand resname might differ
    # find the non-protein, non-lipid, non-water residue
    prot_lip_wat = set("ALA ARG ASN ASP CYS GLN GLU GLY HIS HID HIE HIP ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL DPPC PA PC PE WAT HOH".split()) | {"Na+","Cl-","SOD","CLA"}
    ligres = [r for r in traj.topology.residues if r.name not in prot_lip_wat]
    print("Ligand candidates:", [(r.name, r.resSeq) for r in ligres])
    lig = traj.topology.select(f"resname {ligres[0].name} and (type O or type N)") if ligres else []

asp_oxy = traj.topology.select("resname ASP and (name OD1 or name OD2)")
print(f"Ligand polar atoms: {len(lig)}, ASP carboxyl oxygens: {len(asp_oxy)}")

if len(lig) and len(asp_oxy):
    pairs = np.array([[l,a] for l in lig for a in asp_oxy])
    d = mdt.compute_distances(traj, pairs)*10  # Å
    dmin = d.min(axis=1)
    print(f"Min ligand–ASP(any) distance: mean {dmin.mean():.2f} Å, <3.5Å in {100*(dmin<3.5).mean():.1f}% frames")
    # Which ASP is closest? map back
    closest_pair = pairs[d.mean(axis=0).argmin()]
    closest_asp = traj.topology.atom(int(closest_pair[1])).residue
    print(f"Closest aspartate: resSeq {closest_asp.resSeq}")

In [ ]:
import mdtraj as mdt, numpy as np
from collections import Counter
traj = mdt.load("ht2a_lev_production.dcd", top="ht2a_lev_minimized.pdb")

lig = traj.topology.select("resname LEV")
if len(lig)==0:
    prot_lip_wat = set("ALA ARG ASN ASP CYS GLN GLU GLY HIS HID HIE HIP ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL DPPC PA PC PE WAT HOH".split()) | {"Na+","Cl-","SOD","CLA"}
    ligres=[r for r in traj.topology.residues if r.name not in prot_lip_wat]
    lig = traj.topology.select(f"resname {ligres[0].name}")
    print("Ligand:", ligres[0].name)

# Protein heavy atoms within 4 Å of ligand, per frame -> contact frequency by residue
prot = traj.topology.select("protein and not type H")
contacts = Counter()
for frame in range(0, traj.n_frames, 5):   # every 5th frame for speed
    pairs = np.array([[l,p] for l in lig for p in prot])
    d = mdt.compute_distances(traj[frame], pairs)[0]*10
    close_res = set()
    for k,dist in enumerate(d):
        if dist < 4.0:
            close_res.add(traj.topology.atom(int(pairs[k][1])).residue)
    for r in close_res:
        contacts[(r.resSeq, r.name)] += 1

n = len(range(0, traj.n_frames, 5))
print(f"\nTop levetiracetam contact residues at 5-HT2A (% of {n} frames):")
for (rs,nm),c in contacts.most_common(12):
    # map resSeq back to original 5-HT2A numbering: orig = resSeq-1 mapped through segments
    print(f"  {nm}{rs}: {100*c/n:.0f}%")

In [ ]:
import pickle
with open("/workspace/5ht2a_system/ht2a_gmx_top.pkl","rb") as f:
    s = pickle.load(f)
standard = set("ALA ARG ASN ASP CYS GLN GLU GLY HIS HID HIE HIP ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL CALA CTYR".split())
prot = [r for r in s.residues if r.name in standard]
orig_numbers = list(range(69,266)) + list(range(313,400))
# resSeq in MDTraj = ParmEd index + 1; ParmEd index = position in prot list
idx_to_orig = {i: orig_numbers[i] for i in range(len(prot))}
for resseq in [228,167,161,88,166,225,171,224,87,84,91]:
    idx = resseq - 1   # MDTraj resSeq is 1-based
    orig = idx_to_orig.get(idx, "?")
    print(f"  resSeq {resseq} ({prot[idx].name}) -> original 5-HT2A residue {orig}")

In [ ]:
import mdtraj as mdt
traj = mdt.load("ht2a_lev_production.dcd", top="ht2a_lev_minimized.pdb")

# Keep only protein + ligand (drop water, lipid, ions)
sel = traj.topology.select("protein or resname LEV")
if len(sel) == 0:
    # ligand resname fallback
    prot_lip_wat = set("DPPC PA PC PE WAT HOH".split()) | {"Na+","Cl-","SOD","CLA"}
    keep_res = [r.name for r in traj.topology.residues if r.name not in prot_lip_wat]
    sel = traj.topology.select("protein or resname " + keep_res[-1])

stripped = traj.atom_slice(sel)

# Save the last frame as a PDB (the equilibrated binding pose) — tiny, easy to view
stripped[-1].save("ht2a_lev_pose.pdb")

# Save a lightweight trajectory (every 25th frame) for watching the dynamics
stripped[::25].save("ht2a_lev_binding_traj.dcd")
stripped[0].save("ht2a_lev_binding_top.pdb")
print("✓ Saved: ht2a_lev_pose.pdb (single pose), ht2a_lev_binding_traj.dcd + _top.pdb (movie)")
print(f"Stripped to {stripped.n_atoms} atoms (from {traj.n_atoms})")

---
## Outputs (per ligand)

`ht2a_<lig>_docked.pdbqt`, `_minimized.pdb`, `_equil.xml`, `_production.dcd`, `_production.log`, `_production_final.xml`, `_quicklook.png`

**Run order:** levetiracetam → brivaracetam → piracetam (change `DRUG` in Cell 2, rerun). Then a comparison notebook across all three, plus the early-vs-late contact analysis to see whether each racetam stays in the orthosteric pocket or drifts toward the 5-HT2A side-extended cavity.

**Key 5-HT2A question:** does levetiracetam engage ASP155 (the aminergic anchor it could NOT use in D2)? And does it hold a stable pocket the way it held HIS393 in D2? The contrast between the two receptors is the heart of the off-target comparison.

In [ ]:
REPLICATE_ID = 2
PROD_NS      = 50.0
PROD_STEPS   = int(PROD_NS * 500_000)

tag = f"rep{REPLICATE_ID}"

with open(f"{PREFIX}_equil.xml") as f:
    st = XmlSerializer.deserialize(f.read())

sim.context.setPeriodicBoxVectors(*st.getPeriodicBoxVectors())
sim.context.setPositions(st.getPositions())
sim.context.setVelocitiesToTemperature(310 * u.kelvin, 4000 + REPLICATE_ID)
sim.currentStep = 0
print(f"{PREFIX} replicate {REPLICATE_ID}: seed {4000+REPLICATE_ID}")

sim.reporters.clear()
sim.reporters.append(app.DCDReporter(f"{PREFIX}_production_{tag}.dcd", 25000))
sim.reporters.append(app.CheckpointReporter(f"{PREFIX}_production_{tag}.chk", 500_000))
sim.reporters.append(app.StateDataReporter(
    f"{PREFIX}_production_{tag}.log", 5000,
    step=True, time=True, potentialEnergy=True, kineticEnergy=True,
    totalEnergy=True, temperature=True, volume=True, density=True,
    speed=True, remainingTime=True, totalSteps=PROD_STEPS, separator=","))
sim.reporters.append(app.StateDataReporter(
    sys.stdout, 100_000, step=True, time=True, temperature=True,
    density=True, speed=True, remainingTime=True,
    totalSteps=PROD_STEPS, separator=" | "))

t0 = time.time()
sim.step(PROD_STEPS)
elapsed = (time.time() - t0) / 3600
print(f"\n✓ {tag} done in {elapsed:.2f} h ({PROD_NS/elapsed:.1f} ns/day)")
sim.saveState(f"{PREFIX}_production_{tag}_final.xml")

In [ ]:
REPLICATE_ID = 3
PROD_NS      = 50.0
PROD_STEPS   = int(PROD_NS * 500_000)

tag = f"rep{REPLICATE_ID}"

with open(f"{PREFIX}_equil.xml") as f:
    st = XmlSerializer.deserialize(f.read())

sim.context.setPeriodicBoxVectors(*st.getPeriodicBoxVectors())
sim.context.setPositions(st.getPositions())
sim.context.setVelocitiesToTemperature(310 * u.kelvin, 4000 + REPLICATE_ID)
sim.currentStep = 0
print(f"{PREFIX} replicate {REPLICATE_ID}: seed {4000+REPLICATE_ID}")

sim.reporters.clear()
sim.reporters.append(app.DCDReporter(f"{PREFIX}_production_{tag}.dcd", 25000))
sim.reporters.append(app.CheckpointReporter(f"{PREFIX}_production_{tag}.chk", 500_000))
sim.reporters.append(app.StateDataReporter(
    f"{PREFIX}_production_{tag}.log", 5000,
    step=True, time=True, potentialEnergy=True, kineticEnergy=True,
    totalEnergy=True, temperature=True, volume=True, density=True,
    speed=True, remainingTime=True, totalSteps=PROD_STEPS, separator=","))
sim.reporters.append(app.StateDataReporter(
    sys.stdout, 100_000, step=True, time=True, temperature=True,
    density=True, speed=True, remainingTime=True,
    totalSteps=PROD_STEPS, separator=" | "))

t0 = time.time()
sim.step(PROD_STEPS)
elapsed = (time.time() - t0) / 3600
print(f"\n✓ {tag} done in {elapsed:.2f} h ({PROD_NS/elapsed:.1f} ns/day)")
sim.saveState(f"{PREFIX}_production_{tag}_final.xml")